### **Import libraries**: 

In [1]:
import pandas as pd
from datetime import datetime, timedelta
import numpy as np
import sys
import fastparquet
import warnings
from IPython.display import clear_output

### **Load Parquet file into a Pandas DataFrame object and calculate mean sentiment, sentiment standard deviation, number of articles, number of strongly negative articles, and sentiment momentum.**

In [2]:
def load_and_calculate(parquet_file_name_param, all_metadata):
    df = pd.read_parquet(parquet_file_name_param)
    tickers = list(set(df.index.get_level_values(0)))
    tickers.sort()


    
    dates = list(set(df.index.get_level_values(1)))
    dates = pd.to_datetime(dates)
    dates = [date.date() for date in dates]

    # print(df.loc["AMD"].index.get_level_values(1))
    
    # #print(dates[:10])

    # sys.exit()

    
    dates = [str(element) for element in dates]
    dates.sort()

    iterables = [tickers, pd.to_datetime(dates)]
    index = pd.MultiIndex.from_product(iterables, names=["Ticker", "Timestamp"])
    df_aggregated = pd.DataFrame(index = index, columns = ["sent_mean", "sent_std", "news_count", "pct_strong_negative", "sent_momentum"])
    df_aggregated.sort_index(inplace = True, level = 1, ascending = False, sort_remaining = False)



    df_aggregated = df_aggregated.loc[["AMD", "AMZN"]]

    tickers = ["AMD", "AMZN"]


    counter = 0
    total = len(set(df.loc["AMD"].index.get_level_values(0))) + len(set(df.loc["AMZN"].index.get_level_values(0)))



    #df.sort_index(inplace = True)

    
    for ticker in tickers:
        #print(ticker)
        df_dates = df.loc[ticker].index.get_level_values(0)
        df_headlineIDs = df.loc[ticker].index.get_level_values(1)

        #print(len(df_headlineIDs))

        for date, headlineID in zip(df_dates, df_headlineIDs):
            clear_output()
            print(str(round((counter/total)*100, 2)) + "% complete. " + str(counter) + " " + str(ticker))
            try:
                sent_diff = df.loc[ticker, date]["pos"].sub(df.loc[ticker, date]["neg"], axis = 0)
                sent_mean_today = sent_diff.mean()
                sent_std_today = sent_diff.std()
                news_count_today = len(sent_diff)
                if news_count_today == 1:
                    sent_std_today = 0
                pct_strong_negative_today = (len(df.loc[ticker, date]["neg"][df.loc[ticker, date]["neg"]>0.7]) / len(df.loc[ticker, date]["neg"]))*100
                try:
                    sent_5_day_avg = np.mean([(df.loc[ticker, pd.Timestamp(date) - timedelta(days = i)]["pos"].sub(df.loc[ticker, pd.Timestamp(date) - timedelta(days = i)]["neg"], axis = 0)).mean() for i in range(5)])
                    sent_momentum = sent_mean_today - sent_5_day_avg
                    df_aggregated.loc[(ticker, date), df_aggregated.columns] = sent_mean_today, sent_std_today, news_count_today, pct_strong_negative_today, sent_momentum
                    
                except KeyError:
                    df_aggregated.loc[(ticker, date), df_aggregated.columns] = sent_mean_today, sent_std_today, news_count_today, pct_strong_negative_today, np.nan
                
            except KeyError:
                traceback.print_exc()
                sys.exit()
                #print("No article was published about " + str(ticker) + " today.")


            #clear_output()
            #print(str(round((counter/total)*100, 2)) + "% complete." + "\n")
        
            # with open("status.txt", "w") as f:
            #     f.writelines(all_metadata)
            #     f.write(str(round((counter/total)*100, 2)) + "% complete." + "\n")

            counter +=1

    df_aggregated[df_aggregated.columns] = df_aggregated[df_aggregated.columns].apply(pd.to_numeric, axis = 1)
    return df_aggregated
    #df_aggregated.index.set_levels(new_time_index level=1, inplace=True)


### **Main program used to collectively execute the whole script.**

In [3]:
def main():

    warnings.simplefilter(action='ignore', category=pd.errors.PerformanceWarning)

    # with open("status.txt", "r+") as f:
    #     content = f.read()
    #     time_now = datetime.now()
    #     start_string = "Phase 3 started on " + time_now.strftime("%d-%m-%Y") + " at " + time_now.strftime("%H:%M:%S") + "." + "\n"
    #     f.write(start_string)
    #     f.seek(0)
    #     all_metadata = f.readlines()
    
    parquet_file_name = "5_year_ticker_headlines_with_finbert_rating.parquet"
    df_with_alt_data_features = load_and_calculate(parquet_file_name, "")

    print(df_with_alt_data_features)
    
    #df_with_alt_data_features.to_parquet("5_year_ticker_headlines_with_finbert_rating_calculated.parquet")


    
    
    # with open("status.txt", "w") as f:
    #     f.writelines(all_metadata)
    #     time_now = datetime.now()
    #     end_string = "Phase 3 completed on " + time_now.strftime("%d-%m-%Y") + " at " + time_now.strftime("%H:%M:%S") + "." + "\n"
    #     f.write(end_string) 
        
main()

107.01% complete. 16328 AMZN
                            sent_mean  sent_std  news_count  \
Ticker Timestamp                                              
AMD    2026-04-23 00:00:00        NaN       NaN         NaN   
       2026-04-23 00:00:00        NaN       NaN         NaN   
       2026-04-23 00:00:00        NaN       NaN         NaN   
       2026-04-23 00:00:00        NaN       NaN         NaN   
       2026-04-23 00:00:00        NaN       NaN         NaN   
...                               ...       ...         ...   
AMZN   2021-04-26 17:59:00   0.111586       0.0         1.0   
       2021-04-25 21:22:29   0.239715       0.0         1.0   
       2021-04-25 21:08:00   0.512413       0.0         1.0   
       2021-04-25 08:53:00   0.157851       0.0         1.0   
       2021-04-23 23:34:27   0.246033       0.0         1.0   

                            pct_strong_negative  sent_momentum  
Ticker Timestamp                                                
AMD    2026-04-23 00: